In [ ]:
!pip install -q fastai timm transformers scipy tacoreader rasterio


In [ ]:
REPO_URL = "https://github.com/ArthurrCr/cloudband.git"
PROJECT_DIR = "/content/cloudband"
BRANCH = "main"

import os
import sys

if not os.path.exists(PROJECT_DIR):
    !git clone --quiet {REPO_URL} {PROJECT_DIR}
else:
    !cd {PROJECT_DIR} && git fetch --quiet origin && git reset --quiet --hard origin/{BRANCH}

SRC_DIR = f"{PROJECT_DIR}/src"
os.chdir(PROJECT_DIR)
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)


In [ ]:
import os
from pathlib import Path

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

import pandas as pd
import torch

from cloudband.colab.session import reload_package, start

reload_package("cloudband")

from cloudband.datasets import cloudsen12 as dataset
from cloudband.models.ocm import PAPER_BACKBONES, build_unet, ocm_search_protocol
from cloudband.models.swin_upernet import build_swin_upernet
from cloudband.train.cache import LocalCache
from cloudband.train.data import build_dataloaders
from cloudband.train.loop import checkpoint_name
from cloudband.train.protocol import ocm_shared_protocol, swin_shared_protocol
from cloudband.train.schedule import phase2_networks, run_everything
from cloudband.train.smoke import probe_micro_batch_size
from cloudband.train.store import (
    RunStore,
    ensure_shared_folder,
    load_model_from_checkpoint,
)


In [ ]:
from google.colab import drive

drive.mount("/content/drive")

DRIVE_FOLDER = Path("/content/drive/MyDrive/cloudband_treino")
ensure_shared_folder(DRIVE_FOLDER)

RESULTS_DIR = DRIVE_FOLDER / "results"
CHECKPOINT_DIR = DRIVE_FOLDER / "checkpoints"

session = start(PROJECT_DIR, require_accelerator=True)

store = RunStore(
    results_dir=RESULTS_DIR,
    checkpoint_dir=CHECKPOINT_DIR,
    source_dir=Path(PROJECT_DIR) / "models",
)

# True only when a run is reported "busy" and the session that held it is gone.
BREAK_LOCKS = False

print(f"shared folder: {DRIVE_FOLDER}  |  this session: {store.session_id}")


In [ ]:
import getpass
import os

import requests

token = os.environ.get("HF_TOKEN")
if not token:
    try:
        from google.colab import userdata

        token = userdata.get("HF_TOKEN")
    except Exception:
        token = None
if not token:
    token = getpass.getpass("Hugging Face token (type 'Read' token, hidden): ").strip()

os.environ.pop("HF_TOKEN", None)
if token:
    reply = requests.get(
        "https://huggingface.co/api/whoami-v2",
        headers={"Authorization": f"Bearer {token}"},
        timeout=30,
    )
    if reply.status_code == 200:
        os.environ["HF_TOKEN"] = token
        print(f"token accepted for the account {reply.json().get('name', '?')}")
    else:
        print(f"token refused (HTTP {reply.status_code}); going on without it")
else:
    print("no token: requests use the shared anonymous quota")
del token


In [ ]:
def load_split(remote_source, split):
    table = dataset.load_remote(remote_source)
    differences = dataset.verify_split_sizes(table)
    if differences:
        print("split sizes differ from documented ones:", differences)
    return dataset.select(table, split=split)


train_l1c = load_split(dataset.REMOTE_L1C, split="train")
train_l2a = load_split(dataset.REMOTE_L2A, split="train")
train_table = pd.concat([train_l1c, train_l2a], ignore_index=True)

valid_l1c = load_split(dataset.REMOTE_L1C, split="validation")
valid_l2a = load_split(dataset.REMOTE_L2A, split="validation")
valid_table = pd.concat([valid_l1c, valid_l2a], ignore_index=True)

test_table = load_split(dataset.REMOTE_L1C, split="test")
test_l2a_table = load_split(dataset.REMOTE_L2A, split="test")

print("train patches:", len(train_table))
print("validation patches:", len(valid_table))
print("test patches, L1C (primary):", len(test_table))
print("test patches, L2A (secondary):", len(test_l2a_table))


In [ ]:
import numpy as np

from cloudband.datasets.cloudsen12 import read_sample as gdal_read_sample
from cloudband.datasets.fast_read import FastReader
from cloudband.pipelines.phase0 import select_rgn

CACHE_DIR = Path("/content/cloudband_cache")
WORKERS = 32

fast = FastReader()

checks = (
    (train_table, 0),
    (train_table, len(train_table) - 1),
    (valid_table, 0),
    (test_table, 0),
    (test_l2a_table, 0),
)
for table, position in checks:
    expected, got = gdal_read_sample(table, position), fast(table, position)
    same = np.array_equal(got.image, select_rgn(expected.image)) and np.array_equal(
        got.annotation, expected.annotation
    )
    assert same, f"the fast reader differs from GDAL at row {position}"
print(f"fast reader matches GDAL on {len(checks)} samples")

cache = LocalCache(CACHE_DIR)
for name, table in (
    ("train", train_table),
    ("validation", valid_table),
    ("test", test_table),
    ("test_l2a", test_l2a_table),
):
    cache.build(name, table, read_sample=fast, workers=WORKERS, bands_selected=True)


In [ ]:
from IPython.display import display

from cloudband.datasets.split_audit import audit_splits
from cloudband.eval.level_similarity import compare_levels, level_pairs, summarize

audit = audit_splits(
    {
        ("L1C", "train"): train_l1c,
        ("L2A", "train"): train_l2a,
        ("L1C", "validation"): valid_l1c,
        ("L2A", "validation"): valid_l2a,
        ("L1C", "test"): test_table,
        ("L2A", "test"): test_l2a_table,
    }
)
print(audit.summary)
assert not audit.leaked_rois, "a ROI appears in more than one split"

LEVEL_PAIRS_SAMPLED = 200

pairs = level_pairs(train_l1c, train_l2a, l2a_offset=len(train_l1c))
if not pairs:
    print("\nno patch id is shared by the L1C and L2A tables; the level comparison is skipped")
else:
    chosen = np.random.default_rng(0).choice(
        len(pairs), size=min(LEVEL_PAIRS_SAMPLED, len(pairs)), replace=False
    )
    level_comparison = compare_levels(
        [pairs[i] for i in sorted(chosen)], train_table, cache.reader("train", train_table)
    )
    print(f"\nL1C against L2A of the same patch, {len(chosen)} training patches")
    print("z_rmse: 0 = same image after normalization, 1.41 = unrelated")
    display(summarize(level_comparison).round(3))


In [ ]:
ocm_protocol = ocm_shared_protocol(learning_rate=1e-4)
swin_protocol = swin_shared_protocol(learning_rate=1e-4)

MICRO_BATCH_CANDIDATES = (8, 4, 2)
NUM_WORKERS = 2

decided = store.load_setting("micro_batch")
if decided is None:
    ocm_micro_batch = min(
        probe_micro_batch_size(
            lambda name=name: build_unet(name), ocm_protocol, MICRO_BATCH_CANDIDATES
        )
        for name in PAPER_BACKBONES
    )
    swin_micro_batch = probe_micro_batch_size(
        build_swin_upernet, swin_protocol, MICRO_BATCH_CANDIDATES
    )
    store.save_setting(
        "micro_batch",
        {
            "ocm": ocm_micro_batch,
            "swin": swin_micro_batch,
            "decided_on": torch.cuda.get_device_name(0),
        },
    )
    print("micro-batch decided and saved:", store.load_setting("micro_batch"))
else:
    ocm_micro_batch, swin_micro_batch = decided["ocm"], decided["swin"]
    for name in PAPER_BACKBONES:
        probe_micro_batch_size(
            lambda name=name: build_unet(name), ocm_protocol, (ocm_micro_batch,)
        )
    probe_micro_batch_size(build_swin_upernet, swin_protocol, (swin_micro_batch,))
    print(f"micro-batch from the shared folder (decided on {decided['decided_on']}): "
          f"OCM {ocm_micro_batch}, Swin {swin_micro_batch}; it fits this GPU")

train_reader = cache.reader("train", train_table)
valid_reader = cache.reader("validation", valid_table)


def make_dataloaders(micro_batch_size, protocol):
    return build_dataloaders(
        train_table=train_table,
        valid_table=valid_table,
        micro_batch_size=micro_batch_size,
        num_workers=NUM_WORKERS,
        read_sample=train_reader,
        valid_read_sample=valid_reader,
        bands_selected=True,
        augmentation=protocol.augmentation if protocol.augment else None,
    )


ocm_dls = make_dataloaders(ocm_micro_batch, ocm_protocol)
swin_dls = make_dataloaders(swin_micro_batch, swin_protocol)


In [ ]:
from IPython.display import display

from cloudband.train.status import Plan, status_report

plans = [
    Plan(
        f"OCM {backbone}",
        ocm_protocol,
        backbone,
        search_protocol=ocm_search_protocol(ocm_protocol, backbone),
    )
    for backbone in PAPER_BACKBONES
]
plans.append(Plan("Swin+UPerNet", swin_protocol))
report = status_report(store, plans)

print(report.summary)
columns = ["state", "epochs", "best_val_loss", "session", "note"]
print("\nLearning-rate search")
display(pd.DataFrame(report.search).set_index(["model", "lr"])[columns])
print("Full runs")
display(pd.DataFrame(report.runs).set_index(["model", "seed"])[columns + ["lr"]])


In [ ]:
networks = phase2_networks(ocm_protocol, swin_protocol, ocm_dls, swin_dls)
outcome = run_everything(networks, store, seeds=ocm_protocol.seeds, break_locks=BREAK_LOCKS)


In [ ]:
names = [
    checkpoint_name(ocm_protocol, seed, backbone)
    for seed in ocm_protocol.seeds
    for backbone in PAPER_BACKBONES
] + [checkpoint_name(swin_protocol, seed) for seed in swin_protocol.seeds]

convergence = pd.DataFrame(
    [summary for summary in (store.convergence(name) for name in names) if summary]
)
if convergence.empty:
    print("no finished run with a recorded loss history yet")
else:
    flagged = int(convergence.still_improving.sum())
    print(f"{flagged} of {len(convergence)} runs were still improving at the end")
convergence


In [ ]:
import gc

from cloudband.eval import phase2 as evaluation
from cloudband.train.protocol import PROTOCOL_REVISION

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
REGNETY, CONVNEXT = PAPER_BACKBONES
TEST_SPLITS = (("test", test_table), ("test_l2a", test_l2a_table))


def run_done(name):
    return store.is_done(name)


def seed_done(seed):
    names = [checkpoint_name(ocm_protocol, seed, backbone) for backbone in PAPER_BACKBONES]
    return all(map(run_done, names + [checkpoint_name(swin_protocol, seed)]))


def load_members(seed):
    def load(builder, name):
        return load_model_from_checkpoint(builder, store.checkpoint_path(name), DEVICE)

    return {
        "regnety": load(lambda: build_unet(REGNETY, pretrained=False),
                        checkpoint_name(ocm_protocol, seed, REGNETY)),
        "convnext": load(lambda: build_unet(CONVNEXT, pretrained=False),
                         checkpoint_name(ocm_protocol, seed, CONVNEXT)),
        "swin": load(lambda: build_swin_upernet(pretrained=False),
                     checkpoint_name(swin_protocol, seed)),
    }


seeds_ready = [s for s in ocm_protocol.seeds if seed_done(s)]
print("seeds finished for every model:", seeds_ready, "of", ocm_protocol.seeds)

counts = {split: {} for split, _ in TEST_SPLITS}
for seed in seeds_ready:
    members = None
    for split, table in TEST_SPLITS:
        path = RESULTS_DIR / f"phase2-{PROTOCOL_REVISION}-counts-{split}-seed{seed}.csv"
        if path.exists():
            counts[split][seed] = evaluation.load_counts(path)
            continue
        members = members or load_members(seed)
        frame = evaluation.score_models(
            cache.samples(split, table), members, {"ocm": ("regnety", "convnext")}, DEVICE
        )
        evaluation.save_counts(frame, path)
        counts[split][seed] = frame
        print(f"seed {seed}, {split}: scored and saved")
    del members
    gc.collect()
    torch.cuda.empty_cache()

for seed, frame in counts["test"].items():
    for model, prefix in (("ocm", "ocm"), ("swin", "swin")):
        legacy = RESULTS_DIR / f"{prefix}_shared_per_scene_boa_seed{seed}.csv"
        if legacy.exists():
            gap = evaluation.legacy_check(frame, model, pd.read_csv(legacy, index_col=0))
            print(f"seed {seed} {model}: largest gap to the earlier per-patch BOA {gap:.2e}")


In [ ]:
from IPython.display import display

if not counts["test"]:
    print("no seed is finished for every model yet")
else:
    tables = evaluation.report(counts["test"])
    print(f"L1C test (primary), seeds {sorted(counts['test'])}")
    for title, table in tables.items():
        print(f"\n{title}")
        display(table.round(4))


In [ ]:
# Same scenes at L2A: a robustness check, never pooled with the L1C result.
if not counts["test_l2a"]:
    print("no seed is finished for every model yet")
else:
    primary_roi = evaluation.paired_comparison(counts["test"], "ocm", "swin", "roi")
    tables_l2a = evaluation.report(counts["test_l2a"])
    print(f"L2A test (secondary), seeds {sorted(counts['test_l2a'])}")
    for title, table in tables_l2a.items():
        print(f"\n{title}")
        display(table.round(4))
    same = (
        np.sign(primary_roi["mean diff"]) == np.sign(tables_l2a["OCM vs Swin, per ROI"]["mean diff"])
    )
    print("\nOCM minus Swin per ROI has the same sign at L1C and L2A:", same.to_dict())


In [ ]:
# Maintenance, off by default. CONFIRM_DISCARD: delete runs trained at another learning
# rate than their search chose. CONFIRM_CLEANUP: delete leftovers in the shared folder.
# DRIVE_TIDY "off", "list" or "trash": stale Drive copies; "list"/"trash" ask a second sign-in.
CONFIRM_DISCARD = False
CONFIRM_CLEANUP = False
DRIVE_TIDY = "off"
ACCEPT_PARTIAL_CURVES = False

from cloudband.colab import drive_tidy

report = status_report(store, plans)
to_discard = [row for row in report.runs if row["at_other_rate"]]
if not to_discard:
    print("nothing to discard: every run is at the rate its search chose")
for row in to_discard:
    print(f"{row['model']} seed {row['seed']}: trained at {row['lr']:g}, "
          f"chosen {row['chosen_lr']:g} ({row['state']}, {row['epochs']})")
    if row["state"] == "running":
        print("   skipped: a session is training it right now")
    elif CONFIRM_DISCARD:
        removed = store.discard(row["name"])
        print(f"   discarded ({len(removed)} files)")
if to_discard and not CONFIRM_DISCARD:
    print("\nnothing was deleted: set CONFIRM_DISCARD = True and run again to discard")

unreadable = []
unused = store.unused_files(skipped=unreadable)
size = lambda path: path.stat().st_size / 1e6
for path, reason in unused:
    print(f"{size(path):9.1f} MB  {path.name}\n             {reason}")
if unreadable:
    print(f"\n{len(unreadable)} entr(ies) the Drive lists but cannot read were skipped (trashed a moment ago?). Wait a minute and run again, or remount the Drive.")
print(f"\n{len(unused)} file(s), {sum(size(p) for p, _ in unused):.1f} MB")
if unused and CONFIRM_CLEANUP:
    print(f"removed {store.remove_unused(unused)} file(s)")
elif unused:
    print("nothing was deleted: set CONFIRM_CLEANUP = True and run again to remove them")

known = {path for path, _ in unused}
print("\nlargest files that stay:")
for path, nbytes in store.largest_files(DRIVE_FOLDER, exclude=known):
    print(f"{nbytes / 1e6:9.1f} MB  {path.relative_to(DRIVE_FOLDER)}")

drive_tidy.run_tidy(
    DRIVE_TIDY,
    store,
    plans,
    (RESULTS_DIR, CHECKPOINT_DIR, store.progress_dir),
    (ocm_protocol.run_id, swin_protocol.run_id),
    accept_partial=ACCEPT_PARTIAL_CURVES,
)
